Cuaderno 1: Carga y Preparación de Datos

En esta primera celda importamos las librerías necesarias y configuramos las variables de entorno, incluyendo las credenciales de MetaTrader 5 y las rutas de los directorios.

In [1]:
import os
import pandas as pd
import numpy as np
import MetaTrader5 as mt5
from dotenv import load_dotenv

print("Importando librerías...")
load_dotenv()
print("Variables de entorno cargadas.")

BASE_DIR = os.path.abspath('')
DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Directorio base configurado en: {BASE_DIR}")
print(f"Directorio de datos configurado en: {DATA_DIR}")

MT5_LOGIN = os.getenv("MT5_LOGIN")
MT5_PASSWORD = os.getenv("MT5_PASSWORD")
MT5_SERVER = os.getenv("MT5_SERVER")
MT5_PATH = os.getenv("MT5_PATH", "C:\\Program Files\\Pepperstone MetaTrader 5\\BOT IA\\terminal64.exe")
SYMBOL = "EURUSD"
print(f"Símbolo objetivo establecido: {SYMBOL}")

Importando librerías...
Variables de entorno cargadas.
Directorio base configurado en: c:\Users\juand\OneDrive\Documentos\ECHELONIX\NO SUBIR\vscode\CUADERNOS JUPYTER
Directorio de datos configurado en: c:\Users\juand\OneDrive\Documentos\ECHELONIX\NO SUBIR\vscode\CUADERNOS JUPYTER\data\raw
Símbolo objetivo establecido: EURUSD


La siguiente función establece la conexión con el terminal de MetaTrader 5, extrae el histórico de velas y estructura la información. Se incluye telemetría paso a paso para verificar el estado de la conexión y la manipulación del DataFrame.

In [2]:
def descargar_datos_m5(cantidad_velas):
    print(f"Iniciando conexión con MT5 para descargar {cantidad_velas} velas de {SYMBOL}...")
    if not mt5.initialize(path=MT5_PATH, login=int(61553256), password="mK5flz-aew", server="mt5-demo01.pepperstone.com"):
        print("Error: No se pudo inicializar la conexión con MetaTrader 5.")
        return pd.DataFrame()
        
    print("Conexión con MT5 establecida correctamente.")
    print("Solicitando datos históricos...")
    rates = mt5.copy_rates_from_pos(SYMBOL, mt5.TIMEFRAME_M5, 0, cantidad_velas)
    
    print("Cerrando conexión con MT5...")
    mt5.shutdown()
    
    if rates is None or len(rates) == 0:
        print("Error: No se recibieron datos de MT5.")
        return pd.DataFrame()
        
    print(f"Se recibieron {len(rates)} velas. Convirtiendo a DataFrame...")
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    
    print("Renombrando columnas y filtrando datos necesarios...")
    df_final = df[['open', 'high', 'low', 'close', 'tick_volume']].rename(
        columns={'open': 'open', 'high': 'high', 'low': 'low', 'close': 'close', 'tick_volume': 'volumen'}
    )
    
    print("Descarga y estructuración de datos completada.")
    return df_final

Aquí calculamos las características cuantitativas. La ejecución imprimirá en pantalla cada operación vectorial que se realice sobre la serie de tiempo para confirmar el avance del cálculo de indicadores.

In [3]:
def calcular_indicadores_base(df, ventana=20):
    print("Iniciando cálculo de indicadores base (Feature Engineering)...")
    df = df.copy()
    
    print("Calculando retornos, medias móviles y desviación estándar...")
    df['retorno'] = df['close'].pct_change()
    df['std'] = df['close'].rolling(ventana).std()
    df['ma'] = df['close'].rolling(ventana).mean()
    
    print("Calculando Z-Score...")
    df['z_score'] = (df['close'] - df['ma']) / (df['std'] + 1e-8)
    
    print("Calculando ATR...")
    df['atr'] = df['high'].rolling(ventana).max() - df['low'].rolling(ventana).min()
    
    print("Calculando EMA 288 y distancia del precio a la EMA...")
    df['ema_288'] = df['close'].ewm(span=288, adjust=False).mean()
    df['dist_ema288'] = (df['close'] - df['ema_288']) / (df['std'] + 1e-8)
    
    print("Calculando picos de volumen...")
    df['vol_ma'] = df['volumen'].rolling(ventana).mean()
    df['vol_spike'] = df['volumen'] / (df['vol_ma'] + 1e-8)
    
    print("Calculando métricas de proporciones de las velas japonesas (mechas y cuerpo)...")
    df['total_range'] = df['high'] - df['low']
    df['body_ratio'] = abs(df['close'] - df['open']) / (df['total_range'] + 1e-8)
    df['lower_wick_ratio'] = (df[['open', 'close']].min(axis=1) - df['low']) / (df['total_range'] + 1e-8)
    df['upper_wick_ratio'] = (df['high'] - df[['open', 'close']].max(axis=1)) / (df['total_range'] + 1e-8)
    
    print("Aplicando filtro temporal de Killzone (07:00 a 16:00)...")
    df['is_killzone'] = np.where((df.index.hour >= 7) & (df.index.hour <= 16), 1, 0)
    
    print("Cálculo de indicadores completado.")
    return df

En esta etapa se construyen las etiquetas objetivo. Cada iteración de las condiciones lógicas de Stop Loss y Take Profit será reportada en consola, al igual que la limpieza final de los datos nulos.

In [4]:
def generar_etiquetas(df):
    print("Iniciando generación de etiquetas (Labels)...")
    df = df.copy()
    
    print("Proyectando máximos y mínimos futuros (Ventana: 12 periodos)...")
    df['future_max'] = df['close'].shift(-12).rolling(12).max()
    df['future_min'] = df['close'].shift(-12).rolling(12).min()
    
    print("Calculando niveles dinámicos de Stop Loss basados en ATR...")
    atr_sl = df['atr'] * 0.6
    
    print("Estableciendo distancias de Take Profit (1.5R) y Stop Loss (1R)...")
    tp_long = df['close'] + (atr_sl * 1.5)
    sl_long = df['close'] - atr_sl
    tp_short = df['close'] - (atr_sl * 1.5)
    sl_short = df['close'] + atr_sl
    
    print("Evaluando condiciones de éxito para posiciones Long y Short...")
    cond_long = (df['future_max'] >= tp_long) & (df['future_min'] > sl_long)
    cond_short = (df['future_min'] <= tp_short) & (df['future_max'] < sl_short)
    
    print("Asignando variable objetivo (Target: 1 si toca TP antes de SL, 0 caso contrario)...")
    df['target'] = np.where(cond_long | cond_short, 1, 0)
    
    print("Filtrando el dataset para conservar únicamente las operaciones dentro de la Killzone...")
    df = df[df['is_killzone'] == 1].copy()
    
    print("Eliminando valores nulos generados por el desplazamiento temporal...")
    df = df.dropna()
    
    print("Generación de etiquetas finalizada.")
    return df

Por último, el bloque de ejecución principal. Orquesta las llamadas a todas las funciones anteriores y finaliza guardando el archivo estructurado.

In [5]:
print("INICIANDO PIPELINE DE PREPROCESAMIENTO")
print("-" * 50)

df_raw = descargar_datos_m5(500000)

if not df_raw.empty:
    print("Procesando Features...")
    df_base = calcular_indicadores_base(df_raw)
    
    print("Generando Targets...")
    df_features = generar_etiquetas(df_base)
    
    ruta_archivo = os.path.join(DATA_DIR, f"{SYMBOL}_M5.parquet")
    print(f"Exportando DataFrame resultante ({len(df_features)} filas) a formato Parquet...")
    df_features.to_parquet(ruta_archivo)
    print(f"¡Proceso finalizado con éxito! Archivo guardado en: {ruta_archivo}")
else:
    print("Fallo crítico: El proceso se detuvo debido a que no se obtuvieron datos crudos de la terminal.")

INICIANDO PIPELINE DE PREPROCESAMIENTO
--------------------------------------------------
Iniciando conexión con MT5 para descargar 500000 velas de EURUSD...
Conexión con MT5 establecida correctamente.
Solicitando datos históricos...
Cerrando conexión con MT5...
Se recibieron 500000 velas. Convirtiendo a DataFrame...
Renombrando columnas y filtrando datos necesarios...
Descarga y estructuración de datos completada.
Procesando Features...
Iniciando cálculo de indicadores base (Feature Engineering)...
Calculando retornos, medias móviles y desviación estándar...
Calculando Z-Score...
Calculando ATR...
Calculando EMA 288 y distancia del precio a la EMA...
Calculando picos de volumen...
Calculando métricas de proporciones de las velas japonesas (mechas y cuerpo)...
Aplicando filtro temporal de Killzone (07:00 a 16:00)...
Cálculo de indicadores completado.
Generando Targets...
Iniciando generación de etiquetas (Labels)...
Proyectando máximos y mínimos futuros (Ventana: 12 periodos)...
Calcul